In [ ]:
import cv2
import mediapipe as mp
import numpy as np

In [ ]:
mp_drawing = mp.solutions.drawing_utilis
mp_holistic = mp.solutions.Holistic

In [ ]:
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    cv2.imshow("B' Curl Count", frame)

    if cv2.waitKey(10) & 0xff == ('q'):
        break
cap.release()
cv2.destroyAllWindows()

In [2]:
def calculate_angle(a,b,c):
    a=np.array(a)
    b=np.array(b)
    c=np.array(c)
    radions = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radions*(180.0)/ np.pi)

    if angle > 180.0:
        angle = 360-180
    return angle

In [ ]:
cap = cv2.VideoCapture(0)
counter = 0
with mp_pose.Pose(min_ditection_confidence = 0.5, min_tracking_confidence= 0.5) as pose:


    while cap.isOpened():
        ret, frame = cap.read()
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose_process(image)
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        #Extracting Landmarks
        try:
            landmarks = results.pose_landmarks.landmark


            shoulder = [landmarks[mp_pose.PoseLandmarks.LEFT_SHOULDER.value].x, landmarks[mp_pose.PoseLandmarks.LEFT_SHOULDERS.value].y]
            elbow = [landmarks[mp_pose.PoseLandmarks.LEFT_ELBOW.value].x, landmarks[mp_pose.PoseLandmarks.LEFT_ELBOWS.value].y]
            wrist = [landmarks[mp_pose.PoseLandmarks.LEFT_WRIST.value].x, landmarks[mp_pose.PoseLandmarks.LEFT_WRISTS.value].y]

            angle = calculate_angle(shoulder,elbow,wrist)
            cv2.putText(image.str(angle), tuple(np.multiply(elbow , [640,400]).astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 0.5 ,(255, 255, 255), 2, cv2.LINE_AA )
            
            #logic count
            if angle > 180:
                stage = "Down"
            if angle < 30 and stage == "Down":
                stage = "up"
                counter += 1
                print(counter)

        except:
            pass

        # Text put of counter in the camera image
        cv2.rectangle(image, (0,0), (225,75), (255,255,0), -1)
        cv2.putText(image, "CNT", (15,12), cv2.FONT_HERSHEY_PLAIN, 0.5, (0,0,0), 1 , cv2.LINE_AA)
        cv2.putText(image, str(counter), (30 ,00), cv2.FONT_HERSHEY_COMPLEX, 1, (255,255,255), 2 , cv2.LINE_AA)
        cv2.putText(image, "STAGE", (65,12), cv2.FONT_HERSHEY_PLAIN, 0.5, (0,0,0), 1, cv2.LINE_AA)
        cv2.putText(image, stage,(60,00), cv2.FONT_HERSHEY_SIMPLEX, 2, (255,255,255), 2, cv2.LINE_AA)


        mp_drawing.draw.landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                  mp_drawing.DrawingSpec(color = (255, 0 , 0), thickness = 1, circle_radius = 1))
        if cv2.waitKey(10) & 0xff == ('q'):
            break
       
cap.release()
cv2.destroyAllWindows()